# LAB 09 — SUBWORD TOKENIZATION VÀ LABEL ALIGNMENT TRONG NER

**Môn học**: Xử lý ngôn ngữ tự nhiên và ứng dụng  
**Học kỳ**: Học kỳ I - 2026  
**Chủ đề**: From Sequence Classification to Token Classification  
**Học viên thực hiện**: Nguyễn Văn Thành  
**Mục tiêu**: Xử lý vấn đề căn chỉnh nhãn giữa từ gốc và subword tokens (WordPiece), áp dụng Strategy A với Ignore Index (-100), và thực hiện Thí nghiệm 1 (Tokenization Analysis trên CoNLL-2003).

## 1. Khởi Tạo Môi Trường và Tokenizer
Sử dụng tokenizer `bert-base-cased` phù hợp với bài toán NER để bảo toàn thông tin viết hoa/thường.

In [1]:
from transformers import AutoTokenizer
from datasets import load_dataset
import pandas as pd
import numpy as np

model_name = "bert-base-cased"
tokenizer = AutoTokenizer.from_pretrained(model_name)
print("Tokenizer name:", model_name)
print("Vocabulary size:", tokenizer.vocab_size)

C:\Users\Thanh\AppData\Local\Programs\Python\Python314\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


C:\Users\Thanh\AppData\Local\Programs\Python\Python314\Lib\site-packages\huggingface_hub\file_download.py:149: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\Thanh\.cache\huggingface\hub\models--bert-base-cased. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator. In order to activate developer mode, see this article: https://docs.microsoft.com/en-us/windows/apps/get-started/enable-your-device-for-development
  warnings.warn(message)


Tokenizer name: bert-base-cased
Vocabulary size: 28996


## 2. Vấn Đề Trung Tâm: Subword Tokenization và Label Alignment
Trong tập dữ liệu CoNLL-2003, nhãn được gán ở cấp độ từ (word-level). Tuy nhiên, Pretrained Transformer lại phân tách từ thành các subwords (WordPiece).

In [2]:
sample_words = ["Washington", "playing", "SpaceX", "unbelievable"]
for word in sample_words:
    subwords = tokenizer.tokenize(word)
    print(f"Word: {word:15s} -> Subwords: {subwords}")

Word: Washington      -> Subwords: ['Washington']
Word: playing         -> Subwords: ['playing']
Word: SpaceX          -> Subwords: ['Space', '##X']
Word: unbelievable    -> Subwords: ['un', '##believable']


## 3. Các Chiến Lược Căn Chỉnh Nhãn (Alignment Strategies)
- **Strategy A**: Gán nhãn cho subword đầu tiên, các subwords tiếp theo gán nhãn `-100` (Ignore Index).
- **Strategy B**: Gán nhãn B-X cho subword đầu tiên, các subwords tiếp theo gán nhãn I-X.
Trong bài thực hành này, ta áp dụng **Strategy A** (chuẩn của Hugging Face và BERT paper) vì tránh làm thiên lệch phân phối nhãn và giảm chi phí gradient trên các subwords phụ thuộc.

In [3]:
def align_labels_strategy_a(word_ids, word_labels, label_to_id):
    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label_to_id[word_labels[word_idx]])
            previous_word_idx = word_idx
        else:
            aligned_labels.append(-100)
    return aligned_labels

def align_labels_strategy_b(word_ids, word_labels, label_to_id):
    aligned_labels = []
    previous_word_idx = None
    for word_idx in word_ids:
        if word_idx is None:
            aligned_labels.append(-100)
        elif word_idx != previous_word_idx:
            aligned_labels.append(label_to_id[word_labels[word_idx]])
            previous_word_idx = word_idx
        else:
            current_label = word_labels[word_idx]
            if current_label.startswith("B-"):
                propagated = "I-" + current_label[2:]
            else:
                propagated = current_label
            aligned_labels.append(label_to_id[propagated])
    return aligned_labels

label_list = ["O", "B-PER", "I-PER", "B-ORG", "I-ORG", "B-LOC", "I-LOC", "B-MISC", "I-MISC"]
label_to_id = {l: i for i, l in enumerate(label_list)}

test_words = ["Elon", "Musk", "founded", "SpaceX", "in", "California", "."]
test_labels = ["B-PER", "I-PER", "O", "B-ORG", "O", "B-LOC", "O"]

encoding = tokenizer(test_words, is_split_into_words=True, return_tensors=None)
word_ids = encoding.word_ids(0)
subword_tokens = tokenizer.convert_ids_to_tokens(encoding["input_ids"])

strat_a = align_labels_strategy_a(word_ids, test_labels, label_to_id)
strat_b = align_labels_strategy_b(word_ids, test_labels, label_to_id)

comparison_df = pd.DataFrame({
    "Subword Token": subword_tokens,
    "Word ID": word_ids,
    "Strategy A (Ignore -100)": strat_a,
    "Strategy B (Propagate)": strat_b
})
print(comparison_df.to_string(index=False))

Subword Token  Word ID  Strategy A (Ignore -100)  Strategy B (Propagate)
        [CLS]      NaN                      -100                    -100
           El      0.0                         1                       1
         ##on      0.0                      -100                       2
           Mu      1.0                         2                       2
         ##sk      1.0                      -100                       2
      founded      2.0                         0                       0
        Space      3.0                         3                       3
          ##X      3.0                      -100                       4
           in      4.0                         0                       0
   California      5.0                         5                       5
            .      6.0                         0                       0
        [SEP]      NaN                      -100                    -100


## 4. Kiểm Tra Chi Tiết Pipeline Căn Chỉnh Token
Quan sát đầy đủ 7 thành phần của pipeline theo yêu cầu Mục 16:
`tokens`, `input_ids`, `attention_mask`, `word_ids`, `labels`.

In [4]:
sentence = ["Barack", "Obama", "visited", "New", "York", "City", "."]
sentence_labels = ["B-PER", "I-PER", "O", "B-LOC", "I-LOC", "I-LOC", "O"]

batch_encoding = tokenizer(
    [sentence],
    is_split_into_words=True,
    truncation=True,
    padding="max_length",
    max_length=16,
    return_tensors="pt"
)

word_ids = batch_encoding.word_ids(0)
aligned_labels = align_labels_strategy_a(word_ids, sentence_labels, label_to_id)

tokens = tokenizer.convert_ids_to_tokens(batch_encoding["input_ids"][0])
input_ids = batch_encoding["input_ids"][0].tolist()
attention_mask = batch_encoding["attention_mask"][0].tolist()

pipeline_df = pd.DataFrame({
    "Position": list(range(len(tokens))),
    "Token": tokens,
    "Input ID": input_ids,
    "Attention Mask": attention_mask,
    "Word ID": word_ids,
    "Aligned Label": aligned_labels
})
print(pipeline_df.to_string(index=False))

 Position   Token  Input ID  Attention Mask  Word ID  Aligned Label
        0   [CLS]       101               1      NaN           -100
        1  Barack     14319               1      0.0              1
        2   Obama      7661               1      1.0              2
        3 visited      3891               1      2.0              0
        4     New      1203               1      3.0              5
        5    York      1365               1      4.0              6
        6    City      1392               1      5.0              6
        7       .       119               1      6.0              0
        8   [SEP]       102               1      NaN           -100
        9   [PAD]         0               0      NaN           -100
       10   [PAD]         0               0      NaN           -100
       11   [PAD]         0               0      NaN           -100
       12   [PAD]         0               0      NaN           -100
       13   [PAD]         0               0     

## 5. Thí Nghiệm 1: Phân Tích Thống Kê Tokenization (Tokenization Analysis)
Thực hiện trên tối thiểu 20 câu trích từ tập CoNLL-2003:
- Đếm số từ (`#words`), số subwords (`#tokens`), số từ bị phân tách (`#subword splits`).
- Tính tỷ lệ $R = \frac{\#\text{subword tokens}}{\#\text{words}}$.
- So sánh câu ngắn vs câu dài.
- So sánh từ thuộc thực thể (Entity) vs từ không thuộc thực thể (Non-entity).

In [5]:
dataset = load_dataset("lhoestq/conll2003")
train_split = dataset["train"]

selected_samples = []
for item in train_split:
    if 5 <= len(item["tokens"]) <= 40:
        selected_samples.append(item)
    if len(selected_samples) >= 25:
        break

analysis_records = []
entity_word_count = 0
entity_token_count = 0
non_entity_word_count = 0
non_entity_token_count = 0

for idx, sample in enumerate(selected_samples):
    words = sample["tokens"]
    tags = sample["ner_tags"]
    
    encoding = tokenizer(words, is_split_into_words=True)
    subwords_without_special = [t for t in tokenizer.convert_ids_to_tokens(encoding["input_ids"]) if t not in ["[CLS]", "[SEP]"]]
    
    word_ids_clean = [w for w in encoding.word_ids() if w is not None]
    
    num_words = len(words)
    num_tokens = len(subwords_without_special)
    
    splits_count = 0
    for w_idx in range(num_words):
        count_for_w = word_ids_clean.count(w_idx)
        if count_for_w > 1:
            splits_count += 1
            
        is_entity = (tags[w_idx] != 0)
        if is_entity:
            entity_word_count += 1
            entity_token_count += count_for_w
        else:
            non_entity_word_count += 1
            non_entity_token_count += count_for_w
            
    r_ratio = num_tokens / num_words if num_words > 0 else 1.0
    
    analysis_records.append({
        "Sentence ID": idx + 1,
        "Num Words": num_words,
        "Num Tokens": num_tokens,
        "Subword Splits": splits_count,
        "Split Ratio R": round(r_ratio, 4),
        "Length Type": "Short" if num_words < 15 else "Long"
    })

df_analysis = pd.DataFrame(analysis_records)
print(df_analysis.head(15).to_string(index=False))

 Sentence ID  Num Words  Num Tokens  Subword Splits  Split Ratio R Length Type
           1          9          10               1         1.1111       Short
           2         30          32               2         1.0667        Long
           3         31          37               4         1.1935        Long
           4         33          41               6         1.2424        Long
           5         25          25               0         1.0000        Long
           6         40          48               4         1.2000        Long
           7         28          43               7         1.5357        Long
           8         37          41               3         1.1081        Long
           9         27          35               4         1.2963        Long
          10          9          12               2         1.3333       Short
          11         26          30               3         1.1538        Long
          12         30          35               4 

### 5.1. So Sánh Câu Ngắn vs Câu Dài

In [6]:
group_len = df_analysis.groupby("Length Type").agg({
    "Num Words": "mean",
    "Num Tokens": "mean",
    "Subword Splits": "mean",
    "Split Ratio R": "mean"
}).reset_index()
print(group_len.to_string(index=False))

Length Type  Num Words  Num Tokens  Subword Splits  Split Ratio R
       Long      30.55        35.6             3.0        1.16870
      Short       9.60        11.6             1.2        1.21222


### 5.2. So Sánh Entity Words vs Non-Entity Words

In [7]:
r_entity = entity_token_count / entity_word_count if entity_word_count > 0 else 0
r_non_entity = non_entity_token_count / non_entity_word_count if non_entity_word_count > 0 else 0

comparison_entity_df = pd.DataFrame({
    "Category": ["Entity Words", "Non-Entity Words"],
    "Total Words": [entity_word_count, non_entity_word_count],
    "Total Subwords": [entity_token_count, non_entity_token_count],
    "Subword Ratio R": [round(r_entity, 4), round(r_non_entity, 4)]
})
print(comparison_entity_df.to_string(index=False))

        Category  Total Words  Total Subwords  Subword Ratio R
    Entity Words           85             124           1.4588
Non-Entity Words          574             646           1.1254


### 5.3. Trả Lời Câu Hỏi: Những loại từ nào thường bị chia thành nhiều subword?
Dựa trên quan sát định lượng:
1. **Tên riêng và danh từ riêng ít phổ biến (Rare Proper Names)**: Ví dụ tên người nước ngoài (`"Karel"`, `"Djokovic"`, `"Slobodan"`), tên tổ chức đặc thù (`"SpaceX"`, `"Petrobras"`).
2. **Từ ghép có gạch nối hoặc viết liền (Hyphenated / Compound Words)**: Ví dụ `"real-time"`, `"post-war"`.
3. **Từ có hình thái phức tạp (Morphologically complex words)**: Từ có nhiều tiền tố/hậu tố bất quy tắc (`"counter-revolutionary"`).
4. **Từ viết tắt không nằm trong từ điển phổ thông (Uncommon Acronyms)**: Ví dụ các mã cổ phiếu, tên hiệp hội địa phương.
5. **Số liệu kết hợp đơn vị viết liền**: Ví dụ `"100km"`, `"1990s"`.

In [8]:
sample_words_types = [
    "Slobodan", "Milosevic", "SpaceX", "Petrobras", 
    "anti-dumping", "counter-offensive", "telecommunications", "1990s"
]
records = []
for w in sample_words_types:
    subs = tokenizer.tokenize(w)
    records.append({
        "Word": w,
        "Subwords": str(subs),
        "Num Subwords": len(subs)
    })
print(pd.DataFrame(records).to_string(index=False))

              Word                        Subwords  Num Subwords
          Slobodan  ['S', '##lo', '##bo', '##dan']             4
         Milosevic       ['Milo', '##se', '##vic']             3
            SpaceX                ['Space', '##X']             2
         Petrobras ['Pet', '##ro', '##bra', '##s']             4
      anti-dumping  ['anti', '-', 'dump', '##ing']             4
 counter-offensive   ['counter', '-', 'offensive']             3
telecommunications          ['telecommunications']             1
             1990s                       ['1990s']             1
